In [1]:
import os, glob, math, time, re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# paths
SEARCH_ROOTS = [
    "/Users/carlimankowski/research/grids_real",
    "/Users/carlimankowski/research/grids",
]
STAR_FILE = "final!_errors.csv"
OUT_FILE  = "idc.csv"

# grids
GRID_FAMILIES = ["MIST", "GARSTEC", "Dartmouth", "YREC"]
PRIMARY_GRIDS = ["MIST"]  # final ages

# solar constants
NU_MAX_SUN = 3090.0
DNU_SUN    = 135.1
TEFF_SUN   = 5772.0
LOGG_SUN   = 4.438

# kiauhoku weights
W_MASS      = 1.0
W_RADIUS    = 0
W_DENS      = 1.0
W_LOGG      = 1.0
W_LOGG_SEIS = 1.0
W_TEFF      = 0
W_MET       = 1.0

# error floors
SIGMA_T_FLOOR    = 80.0
SIGMA_MET_FLOOR  = 0.10
FRAC_SYS_NUMAX   = 0.01
FRAC_SYS_DNU     = 0.01
FRAC_M_FLOOR     = 0.06
SIGMA_LOGG_FLOOR = 0.15

# speed knobs
APPLY_MASS_PREFILTER   = True
MASS_WINDOW_K_SEQUENCE = [3.0, 6.0, 12.0]
TOP_PER_GRID           = 4000

# tau_ms prior
AGE_PROP_WIDTH_FRAC = 0.9
TAU_MS_EXPONENT     = 3.3

# helpers
_num = lambda x: pd.to_numeric(x, errors="coerce")

def _nanmean_pair(a, b):
    a = float(a) if pd.notna(a) else np.nan
    b = float(b) if pd.notna(b) else np.nan
    if np.isfinite(a) and np.isfinite(b):
        return 0.5 * (abs(a) + abs(b))
    if np.isfinite(a):
        return abs(a)
    if np.isfinite(b):
        return abs(b)
    return np.nan

def _norm(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")

def tau_ms_from_mass(M, expo=TAU_MS_EXPONENT):
    return 10.0 * (M ** (-expo))

def _tau_band_from_mass(M, index, frac=AGE_PROP_WIDTH_FRAC, expo=TAU_MS_EXPONENT):
    if not np.isfinite(M) or M <= 0.6:
        return (0.005, 14.0)
    tau = tau_ms_from_mass(M, expo=expo)
    w   = frac * tau
    if index < 13:
        return (max(0.05, tau - w), min(14.0, tau + w))
    if 13 <= index < 25:
        return (0.1, min(2.0, tau + w))
    return (max(0.05, tau - w), min(14.0, tau + w))

def _discover_parquets():
    fams = {fam: [] for fam in GRID_FAMILIES}
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for pat in ("*.parquet", "*.pqt"):
            for p in glob.glob(os.path.join(root, "**", pat), recursive=True):
                pl = p.lower()
                if "mist" in pl and "MIST" in GRID_FAMILIES:
                    fams["MIST"].append(p)
                if "garstec" in pl and "GARSTEC" in GRID_FAMILIES:
                    fams["GARSTEC"].append(p)
                if ("dartmouth" in pl or "dart" in pl) and "Dartmouth" in GRID_FAMILIES:
                    fams["Dartmouth"].append(p)
                if "yrec" in pl and "YREC" in GRID_FAMILIES:
                    fams["YREC"].append(p)

    chosen = {}
    for fam, paths in fams.items():
        if not paths:
            continue
        paths.sort(
            key=lambda p: ("eep" in p.lower(), os.path.getsize(p) if os.path.exists(p) else 0),
            reverse=True,
        )
        chosen[fam] = paths[0]

    if not chosen:
        raise FileNotFoundError(f"no parquet grids found in {SEARCH_ROOTS}")

    print("discovered parquets:")
    for fam, path in chosen.items():
        print(f"  {fam}: {path}")
    return chosen

def _peek_names(pf: pq.ParquetFile):
    tbl = pf.read_row_groups([0])
    cols = list(tbl.to_pandas().columns)
    nmap = {_norm(c): c for c in cols}
    return cols, nmap

def _pick(nmap, exact_lists, contains_lists=None):
    for names in exact_lists:
        for nm in names:
            if nm in nmap:
                return nmap[nm]
    if contains_lists:
        for subs in contains_lists:
            for k, orig in nmap.items():
                if all(s in k for s in subs):
                    return orig
    return None

def _column_mapping(cols, nmap):
    age  = _pick(nmap, [[_norm("star_age")], [_norm("age_gyr")], [_norm("age")]], [["age"]])
    mass = _pick(nmap, [[_norm("star_mass")], [_norm("current_mass")], [_norm("mass")]], [["mass"]])

    numax_col = _pick(nmap, [[_norm("nu_max")], [_norm("numax")]], [["nu", "max"]])
    dnu_col   = _pick(nmap, [[_norm("delta_nu")], [_norm("dnu")]], [["delta", "nu"], ["dnu"]])
    teff = _pick(nmap, [[_norm("log_teff")], [_norm("teff")]], [["teff"]])
    logR = _pick(nmap, [[_norm("log_r")], [_norm("log_radius")]], [["log", "r"]])
    logg = _pick(nmap, [[_norm("log_g")], [_norm("logg")]], [["log", "g"]])
    logL = _pick(nmap, [[_norm("log_l")], [_norm("log_lum")]], [["log", "l"]])
    met  = _pick(nmap, [[_norm("fe_h")], [_norm("[fe/h]")], [_norm("feh")]], [["fe", "h"]])

    has_geom_like = (logR is not None) or (logg is not None) or (logL is not None)
    if age is None or (mass is None and not has_geom_like):
        return None

    return {"age": age, "mass": mass, "numax": numax_col, "dnu": dnu_col,
            "teff": teff, "logR": logR, "logg": logg, "logL": logL, "met": met}

def _age_to_gyr(series, name):
    s  = _num(series)
    nm = _norm(name)

    if ("log10_age_yr" in nm) or ("log_age_yr" in nm) or ("logage" in nm and "yr" in nm):
        return (10.0 ** s) / 1e9
    if "logage" in nm or "log_age" in nm:
        return (10.0 ** s) / 1e9

    med = np.nanmedian(s)
    if med > 1e6:
        return s / 1e9
    if med > 100:
        return s / 1e3
    return s

def _iter_rowgroups(pf, usecols):
    nrg = pf.metadata.num_row_groups
    for i in range(nrg):
        tbl = pf.read_row_groups([i], columns=usecols)
        yield tbl.to_pandas()

def _seismic_summary(nu, s_nu, dnu, s_dnu, Teff, s_T, c_numax=1.0, c_dnu=1.0, f_dnu=1.0):
    nu_c  = c_numax * nu
    dnu_c = c_dnu * f_dnu * dnu

    f_nu = np.hypot(s_nu / max(abs(nu), 1e-12), FRAC_SYS_NUMAX)
    f_d  = np.hypot(s_dnu / max(abs(dnu), 1e-12), FRAC_SYS_DNU)
    f_T  = s_T / max(abs(Teff), 1e-12)

    M = (nu_c / NU_MAX_SUN)**3 * (dnu_c / DNU_SUN)**(-4) * (Teff / TEFF_SUN)**1.5
    sigma_lnM = np.sqrt((3*f_nu)**2 + (4*f_d)**2 + (1.5*f_T)**2)
    sM = abs(M) * sigma_lnM

    R = (nu_c / NU_MAX_SUN) * (dnu_c / DNU_SUN)**(-2) * (Teff / TEFF_SUN)**0.5
    sigma_lnR = np.sqrt((f_nu)**2 + (2*f_d)**2 + (0.5*f_T)**2)
    sR = abs(R) * sigma_lnR

    rho = (dnu_c / DNU_SUN)**2
    s_rho = abs(rho) * 2 * f_d

    return M, sM, R, sR, rho, s_rho

def _logg_seismic(nu, s_nu, Teff, s_T, c_numax=1.0):
    nu_c = c_numax * nu
    lg   = LOGG_SUN + math.log10((nu_c / NU_MAX_SUN) * (Teff / TEFF_SUN)**0.5)
    f_nu = s_nu / max(abs(nu), 1e-12)
    f_T  = s_T / max(abs(Teff), 1e-12)
    sig_log10 = (1.0 / np.log(10)) * np.hypot(f_nu, 0.5 * f_T)
    return lg, max(sig_log10, 0.03)

def _huber(residual, sigma, delta=2.0):
    z = np.abs(residual) / max(sigma, 1e-12)
    return np.where(z <= delta, 0.5 * z**2, delta * (z - 0.5 * delta))

def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not mask.any():
        return np.nan
    v = values[mask]
    w = weights[mask]
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    cdf = np.cumsum(w) / np.sum(w)
    return float(v[np.searchsorted(cdf, 0.5)])

# fitter
def _fit_star_one_grid_masscentric(
    pf, cm,
    M_seis, sM_eff,
    R_seis, sR,
    rho_seis, s_rho,
    lg_o, s_lg,
    teff_o, s_T,
    feh_o, s_feh,
    lg_seis, s_lg_seis,
    mass_center, sM_prop,
    age_band, kM,
    apply_mass_prefilter=True,
):
    need = [cm["age"]]
    for c in [cm["mass"], cm["logR"], cm["logL"], cm["logg"], cm["teff"], cm["met"]]:
        if c:
            need.append(c)
    need = list(dict.fromkeys(need))

    kept = []
    scanned = 0
    t0 = time.time()
    lowM, highM = mass_center - kM * sM_prop, mass_center + kM * sM_prop

    for chunk in _iter_rowgroups(pf, usecols=need):
        scanned += len(chunk)
        msk = pd.Series(True, index=chunk.index)

        if age_band is not None:
            ag = _age_to_gyr(chunk[cm["age"]], cm["age"])
            msk &= _num(ag).between(age_band[0], age_band[1])

        if apply_mass_prefilter and (cm["mass"] in chunk.columns or
                                     (cm["logg"] in chunk.columns and cm["logR"] in chunk.columns)):
            if cm["mass"] in chunk.columns:
                M_pf = _num(chunk[cm["mass"]])
            else:
                R_pf  = 10.0 ** _num(chunk[cm["logR"]])
                lg_pf = _num(chunk[cm["logg"]])
                M_pf  = (10.0 ** (lg_pf - LOGG_SUN)) * (R_pf ** 2)
            msk &= _num(M_pf).between(lowM, highM)

        sub = chunk.loc[msk]
        if sub.empty:
            continue

        age_gyr = _age_to_gyr(sub[cm["age"]], cm["age"])

        if cm["mass"] in sub.columns:
            M_mod = _num(sub[cm["mass"]])
        else:
            R_m  = 10.0 ** _num(sub[cm["logR"]]) if cm["logR"] in sub.columns else np.nan
            lg_m = _num(sub[cm["logg"]]) if cm["logg"] in sub.columns else np.nan
            M_mod = (10.0 ** (lg_m - LOGG_SUN)) * (R_m ** 2)

        R_m = 10.0 ** _num(sub[cm["logR"]]) if cm["logR"] in sub.columns else np.nan
        rho_mod = M_mod / (R_m ** 3) if np.isfinite(R_m).any() and np.isfinite(M_mod).any() else np.nan

        chi2 = pd.Series(0.0, index=sub.index)

        if np.isfinite(M_mod).any():
            chi2 += W_MASS * _huber(M_mod - M_seis, sM_eff)

        if np.isfinite(rho_seis) and W_DENS > 0 and np.isfinite(rho_mod).any():
            chi2 += W_DENS * _huber(rho_mod - rho_seis, s_rho)

        if np.isfinite(R_seis) and W_RADIUS > 0 and cm["logR"] in sub.columns:
            chi2 += W_RADIUS * _huber(R_m - R_seis, sR)

        if W_LOGG_SEIS > 0 and cm["logg"] in sub.columns:
            lg_m = _num(sub[cm["logg"]])
            chi2 += W_LOGG_SEIS * ((lg_m - lg_seis) ** 2) / (s_lg_seis ** 2)

        if W_TEFF > 0 and cm["teff"] in sub.columns:
            tname  = cm["teff"]
            is_log = "log" in _norm(tname)
            teff_m = 10.0 ** _num(sub[tname]) if is_log else _num(sub[tname])
            chi2  += W_TEFF * ((teff_m - teff_o) ** 2) / (s_T ** 2)

        if W_MET > 0 and cm["met"] in sub.columns and np.isfinite(feh_o):
            met_m = _num(sub[cm["met"]])
            chi2 += W_MET * ((met_m - feh_o) ** 2) / (s_feh ** 2)

        preds = pd.DataFrame({"age_gyr": age_gyr, "chi2": chi2})
        preds = preds.replace([np.inf, -np.inf], np.nan).dropna()
        if preds.empty:
            continue

        if len(preds) > TOP_PER_GRID * 2:
            preds = preds.nsmallest(TOP_PER_GRID * 2, "chi2")

        kept.append(preds)
        if sum(len(x) for x in kept) >= TOP_PER_GRID * 6:
            break

    if not kept:
        return None, scanned, time.time() - t0

    pool = pd.concat(kept, ignore_index=True)
    best = pool.nsmallest(TOP_PER_GRID, "chi2")
    return best, scanned, time.time() - t0

# load stars
stars = pd.read_csv(STAR_FILE)

required_cols = [
    "TICID",
    "cluster_name",
    "numax_corr", "numax_error_upper", "numax_error_lower",
    "dnu_corr",   "dnu_error_upper",   "dnu_error_lower",
    "cat_teff", "cat_teff_error_upper", "cat_teff_error_lower",
    "cat_met", "cat_met_err",
]
missing = [c for c in required_cols if c not in stars.columns]
if missing:
    raise RuntimeError(f"missing required columns in {STAR_FILE}: {missing}")

# seismic inputs
stars["nu_use"]  = _num(stars["numax_corr"])
stars["dnu_use"] = _num(stars["dnu_corr"])

stars["s_nu"]  = [_nanmean_pair(u, l) for u, l in zip(stars["numax_error_upper"], stars["numax_error_lower"])]
stars["s_dnu"] = [_nanmean_pair(u, l) for u, l in zip(stars["dnu_error_upper"], stars["dnu_error_lower"])]

stars.loc[~np.isfinite(stars["s_nu"])  | (stars["s_nu"]  <= 0), "s_nu"]  = 0.05 * stars["nu_use"].abs()
stars.loc[~np.isfinite(stars["s_dnu"]) | (stars["s_dnu"] <= 0), "s_dnu"] = 0.01 * stars["dnu_use"].abs()

stars["teff_use"] = _num(stars["cat_teff"])
stars.loc[~np.isfinite(stars["teff_use"]), "teff_use"] = TEFF_SUN  # fallback teff

stars["s_T"] = [_nanmean_pair(u, l) for u, l in zip(stars["cat_teff_error_upper"], stars["cat_teff_error_lower"])]
stars.loc[~np.isfinite(stars["s_T"]) | (stars["s_T"] <= 0), "s_T"] = SIGMA_T_FLOOR

stars["feh_use"] = _num(stars["cat_met"])
stars["s_feh"]   = _num(stars["cat_met_err"]).abs()
stars.loc[~np.isfinite(stars["s_feh"]) | (stars["s_feh"] <= 0), "s_feh"] = SIGMA_MET_FLOOR

use_met = True

mok = (stars["nu_use"].notna() & stars["dnu_use"].notna() & (stars["nu_use"] > 0) & (stars["dnu_use"] > 0))
stars_use = stars.loc[mok].copy().reset_index(drop=True)

# derived seismic params
M_seis_arr, sM_eff_arr = [], []
R_seis_arr, sR_arr     = [], []
rho_arr, s_rho_arr     = [], []
lg_seis_arr, s_lg_seis_arr = [], []

for _, r in stars_use.iterrows():
    M, sM, R, sR, rho, s_rho = _seismic_summary(
        float(r["nu_use"]),  float(r["s_nu"]),
        float(r["dnu_use"]), float(r["s_dnu"]),
        float(r["teff_use"]), float(r["s_T"]),
    )
    lg_seis, s_lg_seis = _logg_seismic(r["nu_use"], r["s_nu"], r["teff_use"], r["s_T"])
    sM_eff = max(sM, FRAC_M_FLOOR * abs(M))

    M_seis_arr.append(M);        sM_eff_arr.append(sM_eff)
    R_seis_arr.append(R);        sR_arr.append(max(sR, 1e-4))
    rho_arr.append(rho);         s_rho_arr.append(max(s_rho, 1e-6))
    lg_seis_arr.append(lg_seis); s_lg_seis_arr.append(max(s_lg_seis, 0.03))

stars_use["M_seis"]             = M_seis_arr
stars_use["M_seis_error_upper"] = sM_eff_arr
stars_use["M_seis_error_lower"] = sM_eff_arr

stars_use["R_seis"]             = R_seis_arr
stars_use["R_seis_error_upper"] = sR_arr
stars_use["R_seis_error_lower"] = sR_arr

stars_use["logg_seis"]        = lg_seis_arr
stars_use["logg_seis_upper"]  = s_lg_seis_arr
stars_use["logg_seis_lower"]  = s_lg_seis_arr

stars_use["sM_eff"]    = sM_eff_arr
stars_use["sR"]        = sR_arr
stars_use["rho_seis"]  = rho_arr
stars_use["s_rho"]     = s_rho_arr
stars_use["lg_seis"]   = lg_seis_arr
stars_use["s_lg_seis"] = s_lg_seis_arr

# grids + mappings
chosen = _discover_parquets()
parq    = {}
colmaps = {}

for fam, path in chosen.items():
    pf = pq.ParquetFile(path)
    cols, nmap = _peek_names(pf)
    cm = _column_mapping(cols, nmap)
    if cm is None:
        print(f"warning: skipping grid '{fam}' (no mapping): {path}")
        continue
    parq[fam]    = pf
    colmaps[fam] = cm
    print(f"{fam} column mapping: {cm}")

if "MIST" not in colmaps:
    raise RuntimeError("no usable mist grid could be mapped")

print("using grids:", list(colmaps.keys()))

# cluster mode
cluster_col = "cluster_name"
print(f"cluster mode activated: {cluster_col}")
clusters = stars_use[cluster_col].unique()
print(f"found {len(clusters)} cluster(s): {clusters}")

# cluster mass summary
cluster_mass  = {}
cluster_smass = {}

for cl in clusters:
    sub = stars_use[stars_use[cluster_col] == cl]
    good  = sub["nu_use"] > 50.0
    Mvals = sub.loc[good, "M_seis"]
    if len(Mvals) < 3:
        Mvals = sub["M_seis"]

    M_med = np.nanmedian(Mvals)
    mad   = np.nanmedian(np.abs(Mvals - M_med))
    sigma_mad = 1.4826 * mad
    sM_cluster = max(sigma_mad, 0.08)  # floor

    cluster_mass[cl]  = float(M_med)
    cluster_smass[cl] = float(sM_cluster)

    print(f" • {cl}: {M_med:.3f} ± {sM_cluster:.3f} M⊙ (N={len(Mvals)})")

stars_use["M_seis_cluster"] = stars_use[cluster_col].map(cluster_mass)
stars_use["sM_cluster"]     = stars_use[cluster_col].map(cluster_smass)

# main fit loop
rows = []

for i in range(len(stars_use)):
    r   = stars_use.iloc[i]
    tic = int(r["TICID"])

    M_seis    = float(r["M_seis_cluster"])
    sM_eff    = float(r["sM_cluster"])
    R_seis    = float(r["R_seis"])
    sR        = float(r["sR"])
    rho_seis  = float(r["rho_seis"])
    s_rho     = float(r["s_rho"])
    lg_seis   = float(r["lg_seis"])
    s_lg_seis = float(r["s_lg_seis"])
    teff_o    = float(r["teff_use"])
    s_T       = float(r["s_T"])

    if use_met and pd.notna(r.get("feh_use")):
        feh_o = float(r["feh_use"])
        s_feh = float(r["s_feh"])
    else:
        feh_o = np.nan
        s_feh = np.nan  # no feh

    age_band = _tau_band_from_mass(M_seis, i)

    pool         = []
    per_grid_mu  = {}
    grid_success = {fam: False for fam in colmaps.keys()}

    for kM in MASS_WINDOW_K_SEQUENCE:
        for fam, pf in parq.items():
            if fam not in colmaps or grid_success.get(fam, False):
                continue

            best, scanned, dt = _fit_star_one_grid_masscentric(
                pf, colmaps[fam],
                M_seis=M_seis, sM_eff=sM_eff,
                R_seis=R_seis, sR=sR,
                rho_seis=rho_seis, s_rho=s_rho,
                lg_o=np.nan, s_lg=np.nan,
                teff_o=teff_o, s_T=s_T,
                feh_o=feh_o, s_feh=s_feh,
                lg_seis=lg_seis, s_lg_seis=s_lg_seis,
                mass_center=M_seis,
                sM_prop=max(sM_eff, 0.10),
                age_band=age_band,
                kM=kM,
                apply_mass_prefilter=APPLY_MASS_PREFILTER,
            )

            kept = 0 if best is None else len(best)
            print(f"[{fam}] {tic} idx={i} Mclus={M_seis:.3f}±{sM_eff:.3f} k={kM} kept={kept} scanned≈{scanned:,}")

            if best is not None and len(best) > 0:
                best = best.copy()
                best["grid"] = fam

                chi2 = best["chi2"].to_numpy()
                wts  = np.exp(-0.5 * (chi2 - chi2.min()))
                wts /= wts.sum()
                per_grid_mu[fam] = float(np.sum(wts * best["age_gyr"]))

                pool.append(best)
                grid_success[fam] = True

        if all(grid_success.values()):
            break

    if pool:
        allc = pd.concat(pool, ignore_index=True)
        if "grid" in allc.columns:
            mask_primary = allc["grid"].isin(PRIMARY_GRIDS)
            use_df = allc.loc[mask_primary].copy() if mask_primary.any() else allc.copy()
        else:
            use_df = allc.copy()

        chi2 = use_df["chi2"].to_numpy()
        w    = np.exp(-0.5 * (chi2 - chi2.min()))
        w   /= w.sum()

        ages = use_df["age_gyr"].to_numpy()
        final_age = weighted_median(ages, w)

        mean_age  = float(np.sum(w * ages))
        final_std = float(np.sqrt(np.sum(w * (ages - mean_age)**2)))
    else:
        final_age = np.nan
        final_std = np.nan  # no fit

    out = {
        "TICID":       tic,
        "age_gyr":     final_age,
        "age_std":     final_std,
        "M_seis_used": M_seis,
        "sM_used":     sM_eff,
        "cluster":     r[cluster_col],
    }

    for fam_label in GRID_FAMILIES:
        out[f"age_{fam_label}"] = per_grid_mu.get(fam_label, np.nan)

    rows.append(out)

result = pd.DataFrame(rows)
result.to_csv(OUT_FILE, index=False)
print(f"cluster mode complete → {OUT_FILE} ({len(result)} stars)")
display(result.head(20))  # quick peek

# sigma analysis
print("running ±σ mass+metal analysis (mist only)")
SIGMA_OUT = "clusterwork_KH_CLUSTER_SIGMAAGES.csv"

def _run_sigma_case(r, M_use, sM_use, Fe_use, sFe_use, index, label):
    tic = int(r["TICID"])

    R_seis    = float(r["R_seis"])
    sR        = float(r["sR"])
    rho_seis  = float(r["rho_seis"])
    s_rho     = float(r["s_rho"])
    lg_seis   = float(r["lg_seis"])
    s_lg_seis = float(r["s_lg_seis"])
    teff_o    = float(r["teff_use"])
    s_T       = float(r["s_T"])

    feh_o = Fe_use
    s_feh = sFe_use

    age_band = _tau_band_from_mass(M_use, index)
    print(f"\nTIC {tic} idx={index} [{label}] M={M_use:.3f}±{sM_use:.3f} FeH={feh_o:.3f}±{s_feh:.3f} age_band={age_band}")

    pool = []
    pergrid = {}

    for fam, pf in parq.items():
        if fam not in colmaps:
            continue

        best, scanned, dt = _fit_star_one_grid_masscentric(
            pf, colmaps[fam],
            M_seis=M_use, sM_eff=sM_use,
            R_seis=R_seis, sR=sR,
            rho_seis=rho_seis, s_rho=s_rho,
            lg_o=np.nan, s_lg=np.nan,
            teff_o=teff_o, s_T=s_T,
            feh_o=feh_o, s_feh=s_feh,
            lg_seis=lg_seis, s_lg_seis=s_lg_seis,
            mass_center=M_use,
            sM_prop=max(sM_use, 0.20),
            age_band=age_band,
            kM=99.0,
            apply_mass_prefilter=False,
        )

        kept = 0 if best is None else len(best)
        print(f"   [{fam}] kept={kept}")

        if best is not None and kept > 0:
            best = best.copy()
            best["grid"] = fam
            chi = best["chi2"].to_numpy()
            w   = np.exp(-0.5 * (chi - chi.min()))
            w  /= w.sum()
            pergrid[fam] = float(np.sum(w * best["age_gyr"]))
            pool.append(best)

    if not pool:
        return np.nan, np.nan, pergrid

    allc = pd.concat(pool, ignore_index=True)

    if (allc["grid"] == "MIST").any():
        use_df = allc.loc[allc["grid"] == "MIST"].copy()
    else:
        use_df = allc.copy()

    chi = use_df["chi2"].to_numpy()
    w   = np.exp(-0.5 * (chi - chi.min()))
    w  /= w.sum()

    ages = use_df["age_gyr"].to_numpy()
    age_med  = weighted_median(ages, w)
    age_mean = float(np.sum(w * ages))
    age_std  = float(np.sqrt(np.sum(w * (ages - age_mean)**2)))

    return age_med, age_std, pergrid

sigma_rows = []

for idx, r in stars_use.iterrows():
    tic = int(r["TICID"])

    M0  = float(r["M_seis_cluster"])
    sM  = float(r["sM_cluster"])
    Fe0 = float(r["feh_use"])
    sFe = float(r["s_feh"])

    age_central = float(result.loc[result["TICID"] == tic, "age_gyr"].values[0])
    std_central = float(result.loc[result["TICID"] == tic, "age_std"].values[0])

    age_m, std_m, per_m = _run_sigma_case(r, M0 - sM, sM, Fe0 - sFe, sFe, idx, label="minusσ")
    age_p, std_p, per_p = _run_sigma_case(r, M0 + sM, sM, Fe0 + sFe, sFe, idx, label="plusσ")

    sigma_rows.append({
        "TICID": tic,
        "age_central":          age_central,
        "age_std_central":      std_central,
        "age_minus1sigma":      age_m,
        "age_std_minus1sigma":  std_m,
        "age_plus1sigma":       age_p,
        "age_std_plus1sigma":   std_p,
        "M0":  M0,  "sM":  sM,
        "Fe0": Fe0, "sFe": sFe,
    })

sigma_df = pd.DataFrame(sigma_rows)
sigma_df.to_csv(SIGMA_OUT, index=False)

print(f"sigma analysis complete → {SIGMA_OUT}")
display(sigma_df.head(20))  # quick peek



/var/folders/w8/mst51j4j6b3664rd72bjyw7c0000gn/T/ipykernel_43972/1673600053.py:397: DtypeWarning: Columns (16,17,18,34,36,38,64,65,80) have mixed types. Specify dtype option on import or set low_memory=False.
  stars = pd.read_csv(STAR_FILE)


Discovered Parquet grids:
  MIST: /Users/carlimankowski/research/grids_real/MIST/mist/mist_eep.pqt
  GARSTEC: /Users/carlimankowski/research/grids_real/GARSTEC/garstec/garstec_eep.pqt
  Dartmouth: /Users/carlimankowski/research/grids_real/Dartmouth/dartmouth/dartmouth_eep.pqt
  YREC: /Users/carlimankowski/research/grids_real/YREC/yrec/yrec_eep.pqt
MIST column mapping: {'age': 'star_age', 'mass': 'star_mass', 'numax': 'nu_max', 'dnu': 'delta_nu', 'teff': 'log_Teff', 'logR': 'log_R', 'logg': 'log_g', 'logL': 'log_L', 'met': None}
GARSTEC column mapping: {'age': 'Age(Myr)', 'mass': None, 'numax': None, 'dnu': None, 'teff': 'Teff', 'logR': None, 'logg': 'logg', 'logL': 'Log L/Lsun', 'met': None}
Dartmouth column mapping: {'age': 'Age (yrs)', 'mass': None, 'numax': None, 'dnu': None, 'teff': None, 'logR': 'Log R', 'logg': 'Log g', 'logL': 'Log L', 'met': None}
YREC column mapping: {'age': 'Age(Gyr)', 'mass': 'mass_conv_core', 'numax': None, 'dnu': None, 'teff': 'Log Teff(K)', 'logR': 'logrh

,TICID,age_gyr,age_std,M_seis_used,sM_used,cluster,age_MIST,age_GARSTEC,age_Dartmouth,age_YREC
0,306345133,0.497309,0.188130,2.428330,0.451342,Theia_6046,0.534594,0.921674,0.808915,0.868990
1,43902016,0.549035,0.169711,2.428330,0.451342,Theia_6046,0.566201,0.887232,0.841260,0.872128
2,24297458,0.522843,0.184930,2.428330,0.451342,Theia_6046,0.552070,0.904612,0.760255,0.895754
3,24666306,0.466710,0.170244,2.428330,0.451342,Theia_6046,0.506930,0.921306,0.774204,0.869666
4,24444542,0.499364,0.169318,2.428330,0.451342,Theia_6046,0.524050,0.897287,0.828101,0.871068
5,249064439,0.728874,0.166945,2.428330,0.451342,Theia_6046,0.727258,0.902735,0.733414,0.893881
6,34472483,0.516288,0.181190,2.428330,0.451342,Theia_6046,0.549027,0.910297,0.757187,0.888026
7,67569102,1.179731,0.563604,1.526456,0.136756,NGC_752,1.351686,2.344661,2.355908,2.094710
8,67420118,1.422647,0.718042,1.526456,0.136756,NGC_752,1.618417,2.347608,2.598952,2.093082
9,186970424,1.448121,0.589820,1.526456,0.136756,NGC_752,1.609623,2.231921,2.732916,2.117565



=== RUNNING ±σ MASS+METAL AGE ANALYSIS (MIST ONLY, INDEX-BASED) ===

--- TIC 306345133 idx=0 [minusσ] M=1.977±0.451 FeH=-0.060±0.010  AGE_BAND=(0.10548411736373009, 2.0041982299108727)
   [MIST] kept=4000
   [GARSTEC] kept=4000
   [Dartmouth] kept=4000
   [YREC] kept=4000

--- TIC 306345133 idx=0 [plusσ] M=2.880±0.451 FeH=-0.040±0.010  AGE_BAND=(0.05, 0.5793242296896292)
   [MIST] kept=4000
   [GARSTEC] kept=3018
   [Dartmouth] kept=4000
   [YREC] kept=2244

--- TIC 43902016 idx=1 [minusσ] M=1.977±0.451 FeH=-0.060±0.010  AGE_BAND=(0.10548411736373009, 2.0041982299108727)
   [MIST] kept=4000
   [GARSTEC] kept=4000
   [Dartmouth] kept=4000
   [YREC] kept=4000

--- TIC 43902016 idx=1 [plusσ] M=2.880±0.451 FeH=-0.040±0.010  AGE_BAND=(0.05, 0.5793242296896292)
   [MIST] kept=4000
   [GARSTEC] kept=3018
   [Dartmouth] kept=4000
   [YREC] kept=2244

--- TIC 24297458 idx=2 [minusσ] M=1.977±0.451 FeH=-0.170±0.120  AGE_BAND=(0.10548411736373009, 2.0041982299108727)
   [MIST] kept=4000
   [GARST

,TICID,age_central,age_std_central,age_minus1sigma,age_std_minus1sigma,age_plus1sigma,age_std_plus1sigma,M0,sM,Fe0,sFe
0,306345133,0.497309,0.188130,0.575272,0.262225,0.386890,0.096096,2.428330,0.451342,-0.05000,0.009637
1,43902016,0.549035,0.169711,0.667043,0.326314,0.419082,0.088899,2.428330,0.451342,-0.05000,0.010000
2,24297458,0.522843,0.184930,0.630673,0.352893,0.404549,0.090868,2.428330,0.451342,-0.05000,0.119868
3,24666306,0.466710,0.170244,0.540896,0.273889,0.389742,0.092401,2.428330,0.451342,-0.05000,0.010000
4,24444542,0.499364,0.169318,0.559611,0.256879,0.397959,0.096030,2.428330,0.451342,-0.05000,0.010000
5,249064439,0.728874,0.166945,1.027437,0.375056,0.472372,0.088529,2.428330,0.451342,-0.05000,0.010000
6,34472483,0.516288,0.181190,0.627362,0.329415,0.409373,0.088950,2.428330,0.451342,-0.05000,0.010000
7,67569102,1.179731,0.563604,1.418117,0.785649,1.007853,0.385431,1.526456,0.136756,-0.17010,0.074278
8,67420118,1.422647,0.718042,1.819612,1.011848,1.110854,0.495062,1.526456,0.136756,-0.08430,0.036027
9,186970424,1.448121,0.589820,1.868020,0.975327,1.269673,0.379385,1.526456,0.136756,-0.06690,0.049074
